# ML Workflow with DVC Workshop

**Course:** CSCN8010 – Foundations of ML Frameworks  

> Important: if `dvc pull` fails in the professor repo, do **not** get stuck there.  
> In this workshop, the correct next step is usually to run the pipeline locally with `dvc repro`.

## Exact step-by-step workflow to follow in VS Code

### Part A — In the VS Code terminal
Open the cloned repo folder in VS Code and run these commands from the **project root**:

```powershell
.

env\Scripts\Activate.ps1
pip install dvc torch torchvision scikit-learn pandas pyyaml jupyter notebook
```

Then check whether the repo is already initialized:

```powershell
dir
dir .dvc
```

If the repo was cloned from the professor, **do not run `git init` or `dvc init` again** unless those folders/files are missing.

If `dvc pull` gives a missing-files error, skip it and run:

```powershell
dvc repro
dvc metrics show
```

If `dvc repro` fails, then debug the actual script error by running:

```powershell
python src\prepare.py
python src	rain.py
```

### Part B — In this notebook
While running the notebook, do these things:

- read the explanation sections
- keep the code templates for `prepare.py`, `train.py`, `predict.py`, `params.yaml`, and `dvc.yaml`
- complete the three challenge sections
- add your own short talking points under each challenge
- keep your writing simple and explainable

### Part C — In the repo files
You will likely need to edit these files during the workshop:

- `params.yaml`
- `src/train.py`
- `src/predict.py`
- `dvc.yaml`
- `README.md`

### What to do after finishing
At the end:

1. save your final notebook
2. update `README.md`
3. run `git status`
4. commit and push to GitHub
5. send the GitHub `.git` repo link as instructed

## Review summary of the current project files

I checked the notebook and the related project files together. These were the main issues I found and corrected in this notebook version:

1. The notebook training code and the actual `src/train.py` file were not matching. I kept the notebook version aligned with the **parameter-driven** approach using `params.yaml`.
2. The actual project file `dvc.yaml` was missing `batch_size` in the tracked parameters, even though the notebook experiments change it. In this notebook, the DVC example keeps `batch_size` included.
3. The original project file `src/train.py` was dividing image values by `255` even though `transforms.ToTensor()` already scales MNIST images. In this notebook, I kept the cleaner version without that extra scaling.
4. The earlier prediction example imported `SimpleCNN` from `src.train`, which can accidentally run training code during import. I replaced that notebook example with a cleaner standalone `predict.py` version.
5. The PowerShell prediction display example was not reliable inside a notebook. I replaced it with a Python-based check that works more consistently.
6. The Git example was trying to add `metrics.json` even though that file is ignored and pipeline-generated. I cleaned those commands in this notebook.

### Important note

Your **notebook content is mostly on the right track**, but the **related project files inside the zip are not fully consistent with it yet**. So the notebook needed cleanup to make the explanation, workflow, and talking points match better from top to bottom.


# ML Workflow with DVC

This notebook is a guided workshop for building a minimal, reproducible machine learning workflow with **Data Version Control (DVC)** using the project structure below.

```text
project/
├── data/
├── src/
│   ├── prepare.py
│   ├── train.py
├── params.yaml
├── dvc.yaml
```

The workflow assumes you are working in Visual Studio Code and running commands from the project root folder.


## 1. Introduction to DVC

**DVC stands for Data Version Control.  
It is an open‑source tool designed for managing machine learning and data science projects, especially when you work with large datasets, models, and experiments that don’t fit well into Git alone.**

In the workplace, DVC is useful when teams need to coordinate code, data, models, and results without placing large binary assets directly inside a Git repository. In many organizations, software engineers, data scientists, and researchers work together across branches, machines, and cloud environments. DVC helps these teams keep a lightweight Git history while storing the actual datasets and trained models in external storage. This makes it easier to reproduce results, compare experiments, onboard new team members, and reduce confusion about which dataset or model version produced a particular result. DVC is especially valuable in research labs, applied ML teams, and MLOps pipelines where traceability and repeatability matter.

**Official DVC portal:** <https://dvc.org/>  
**Official documentation:** <https://doc.dvc.org/>


## 2. What is the role of DVC in the MLOps workflow?

DVC sits at the intersection of **code versioning, data management, experiment tracking, and reproducibility**. In a modern MLOps workflow, it helps connect what was run, on which data, with which parameters, and what outputs were produced.

### What problem does DVC solve?

Git works very well for **code**, but machine learning projects often include assets that Git alone does not handle elegantly:

- Large datasets
- Model checkpoints and trained models
- Intermediate artifacts
- Repeatable experiment pipelines
- Clear traceability between code, data, and results

DVC complements Git by versioning the **metadata** about large files and pipeline outputs while storing the heavy artifacts elsewhere.

### What DVC does

DVC helps you:

#### ✅ Version datasets and models
- Keeps lightweight metadata (`.dvc` files) in Git
- Stores actual data in external storage (local disk, S3, Azure Blob, GCP, SSH, etc.)

#### ✅ Track experiments
- Records parameters, metrics, and outputs
- Allows you to compare experiments (like a Git diff for ML)

#### ✅ Ensure reproducibility
- Defines pipelines so models can be rebuilt from raw data
- Tracks dependencies between data, code, and models

#### ✅ Collaborate effectively
- Team members can reproduce results exactly
- Everyone gets the same data/model versions tied to a Git commit


## 3. Build a Minimal DVC Project

In this section, students build and run a small DVC project for a CNN trained on MNIST.

### Learning goals

By the end of this section, students should be able to:

- initialize a Git + DVC project
- understand the role of each project file
- run a two-stage DVC pipeline
- inspect metrics
- make a controlled experiment change
- explain why DVC reruns some stages and skips others


### 3.1 Setup commands (Bash)

In [1]:
# Run these in a Bash terminal from the project root

# pip install dvc torch torchvision scikit-learn pandas pyyaml
# git init
# dvc init


#### Talking points

- This part is only for the very first setup of the project folder.
- Here I install the main libraries and initialize both Git and DVC.
- The main idea is that Git handles code history, while DVC helps track ML data, models, and pipeline steps.


**Explanation**

- `pip install ...` installs the Python packages used in this workshop.
- `git init` creates a Git repository so code and DVC metadata can be versioned.
- `dvc init` creates the DVC configuration inside the repository.


### 3.2 Setup commands (PowerShell)

### Important note for a cloned professor repo

The setup commands shown below are useful for understanding the workflow, but for a **cloned repo** you usually do this:

- activate the virtual environment
- install the packages
- **skip re-running** `git init` and `dvc init` if the repo already has `.git`, `.dvc`, `dvc.yaml`, and related files
- if `dvc pull` fails because remote outputs are missing, use `dvc repro` to recreate the outputs locally

In [2]:
# Run these in PowerShell from the project root

# pip install dvc torch torchvision scikit-learn pandas pyyaml
# git init
# dvc init


#### Talking points

- These are the same setup steps, but written for PowerShell since I am working in Windows and VS Code.
- I need to run these from the project root so all project files are created in the correct place.
- Once this is done, the folder is ready for the DVC workflow.


The commands are the same in PowerShell for this basic setup. Later, file paths may look slightly different on Windows.


### 3.3 Project structure

```text
project/
├── data/
├── src/
│   ├── prepare.py
│   ├── train.py
├── params.yaml
├── dvc.yaml
```

### What each part does

- `data/` stores raw and processed data artifacts.
- `src/prepare.py` downloads and prepares the dataset.
- `src/train.py` trains the CNN and writes outputs for the pipeline.
- `params.yaml` stores experiment settings such as epochs and learning rate.
- `dvc.yaml` defines the DVC pipeline stages, their dependencies, and their outputs.


### 3.4 Create `params.yaml`

In [3]:
epochs: 2
lr: 0.001
batch_size: 64


#### Talking points

- I used `params.yaml` to keep experiment settings outside the training script.
- This makes the workflow cleaner because I can change epochs, learning rate, or batch size without rewriting the code.
- It also makes DVC experiments easier to compare later.


This file stores experiment parameters outside the training code. That makes it easier to rerun the pipeline after changing one setting and observe the result.


### 3.5 `src/prepare.py`

In [4]:
import os
import torch
from torchvision import datasets, transforms

OUTPUT_DIR = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root="data/raw",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="data/raw",
    train=False,
    download=True,
    transform=transform
)

def dataset_to_tensors(dataset):
    images = []
    labels = []

    for img, label in dataset:
        images.append(img)
        labels.append(label)

    images = torch.stack(images)
    labels = torch.tensor(labels)

    return images, labels

train_images, train_labels = dataset_to_tensors(train_dataset)
test_images, test_labels = dataset_to_tensors(test_dataset)

torch.save((train_images, train_labels), os.path.join(OUTPUT_DIR, "train.pt"))
torch.save((test_images, test_labels), os.path.join(OUTPUT_DIR, "test.pt"))

print("Data preparation complete.")
print(f"Saved to: {OUTPUT_DIR}")


Data preparation complete.
Saved to: data/processed


#### Talking points

- This script prepares the MNIST dataset and saves the processed tensors into the `data/processed` folder.
- I used `ToTensor()` so the images are already converted into PyTorch tensors in a usable format.
- The output of this stage becomes the input for the training stage, so this is the starting point of the pipeline.


### Role of `prepare.py` in the pipeline

This script is the **data preparation stage**. It:

- downloads MNIST
- converts the dataset into tensors
- saves the processed training and test data to disk

For teaching purposes, this script is intentionally simple. It avoids complex preprocessing so students can focus on the workflow and the role of DVC.


### 3.6 `src/train.py`

In [5]:
import os
import json
import yaml
import torch
import torch.nn as nn
import torch.optim as optim

DATA_DIR = "data/processed"
MODEL_PATH = "model.pt"
METRICS_PATH = "metrics.json"

with open("params.yaml", "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

EPOCHS = params["epochs"]
LR = params["lr"]
BATCH_SIZE = params["batch_size"]

train_images, train_labels = torch.load(os.path.join(DATA_DIR, "train.pt"))
test_images, test_labels = torch.load(os.path.join(DATA_DIR, "test.pt"))

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(8 * 13 * 13, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

model.train()
for epoch in range(EPOCHS):
    for i in range(0, len(train_images), BATCH_SIZE):
        x_batch = train_images[i:i + BATCH_SIZE]
        y_batch = train_labels[i:i + BATCH_SIZE]

        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {loss.item():.4f}")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    outputs = model(test_images)
    _, predicted = torch.max(outputs, 1)
    total += test_labels.size(0)
    correct += (predicted == test_labels).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

torch.save(model.state_dict(), MODEL_PATH)

metrics = {"accuracy": round(accuracy, 4)}
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

print(f"Model saved to {MODEL_PATH}")
print(f"Metrics saved to {METRICS_PATH}")


Epoch 1/5, Loss: 0.0894
Epoch 2/5, Loss: 0.0443
Epoch 3/5, Loss: 0.0351
Epoch 4/5, Loss: 0.0339
Epoch 5/5, Loss: 0.0324
Test Accuracy: 0.9739
Model saved to model.pt
Metrics saved to metrics.json


#### Talking points

- This training script reads the experiment settings from `params.yaml`, so the run is controlled by tracked parameters.
- The model is a small CNN, which is enough for showing the DVC workflow without making the example too complex.
- At the end, the script saves both the trained model and the accuracy metric, which DVC can track and compare.


### Role of `train.py` in the pipeline

This script is the **training stage**. It:

- loads the processed tensors
- reads hyperparameters from `params.yaml`
- trains a tiny CNN
- saves a trained model to `model.pt`
- writes evaluation results to `metrics.json`

In this workshop, the model is intentionally small so that students can run it quickly on a CPU and focus on reproducibility rather than model complexity.


### 3.7 `dvc.yaml`

stages:
  prepare:
    cmd: python src/prepare.py
    deps:
      - src/prepare.py
    outs:
      - data/processed

  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/processed
    params:
      - epochs
      - lr
      - batch_size
    outs:
      - model.pt
    metrics:
      - metrics.json


### How to read `dvc.yaml`

- `stages` defines the pipeline.
- `prepare` is the first stage and runs `python src/prepare.py`.
- `deps` lists dependencies. If one changes, DVC knows the stage may need to rerun.
- `outs` lists outputs produced by a stage.
- `train` depends on both `src/train.py` and the prepared data directory.
- `params` tells DVC which experiment settings should be tracked from `params.yaml`.
- `metrics` tells DVC which result files should be displayed and compared.

This file is the heart of the pipeline: it defines **what runs, in what order, based on which inputs**.


### What to do if `dvc pull` fails

A failed `dvc pull` in this workshop usually means the remote storage does not contain all tracked outputs yet.  
That does **not** automatically mean your setup is wrong.

In that case, do this instead:

```powershell
dvc repro
dvc metrics show
```

This recreates the pipeline outputs on your own machine.

### 3.8 Run the pipeline

In [6]:
# Bash

!python src/prepare.py
!python src/train.py


Data preparation complete.
Saved to: data/processed
Epoch 1/2, Loss: 1.3445
Epoch 2/2, Loss: 0.5489
Test Accuracy: 0.8491
Model saved to model.pt
Metrics saved to metrics.json


#### Talking points

- Here I run the scripts directly once so I can understand what each file is doing before I depend on DVC.
- `prepare.py` creates the processed data, and `train.py` trains the model and writes the metric file.
- This step is useful for checking that the code itself works before running the full pipeline.


Optional warm-up: run the scripts directly once so students can see what each stage does individually.

In [7]:
# Bash

!dvc repro
!dvc metrics show


Stage 'prepare' didn't change, skipping
Stage 'train' is cached - skipping run, checking out outputs

Stage 'predict' didn't change, skipping
Use `dvc push` to send your updates to remote storage.
Path          accuracy
metrics.json  0.8242


#### Talking points

- `dvc repro` runs only the stages that need to be run based on dependencies and changes.
- `dvc metrics show` lets me quickly read the saved metric from the pipeline output.
- This is where DVC starts to feel useful because I do not have to rerun everything manually every time.


In [8]:
# PowerShell


!dvc repro
!dvc metrics show


Stage 'prepare' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'predict' didn't change, skipping
Data and pipelines are up to date.
Path          accuracy
metrics.json  0.8242


#### Talking points

- This is the same pipeline check again, just shown in a PowerShell-friendly form.
- I included it because my local setup is Windows, so these are the commands I would actually use.
- The goal is still the same: reproduce the pipeline and inspect the saved metric.


### What students should notice

- DVC runs the stages in dependency order.
- The pipeline creates `data/processed`, `model.pt`, and `metrics.json`.
- `dvc metrics show` reads the accuracy directly from the metrics file.


### 3.9 Example metrics file

In [9]:
{
  "accuracy": 0.6967
}


{'accuracy': 0.6967}

#### Talking points

- This is a simple example of what the metric file looks like after training finishes.
- Keeping the metric in JSON format is helpful because DVC can read it directly.
- It also makes experiment comparison easier later.


### 3.10 Make controlled changes to the experiment

The idea is to change **one thing at a time** and then rerun the pipeline.

#### Example A: Increase the number of epochs


In [10]:
epochs: 5
lr: 0.001
batch_size: 64


#### Talking points

- In this example I change only the number of epochs.
- I am doing one controlled change at a time so I can understand what caused any result difference.
- That is better than changing many things together and then not knowing what actually helped.


In [11]:
# Bash or PowerShell

!dvc repro
!dvc metrics show
!dvc metrics diff


Stage 'prepare' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'predict' didn't change, skipping
Data and pipelines are up to date.
Path          accuracy
metrics.json  0.8242


#### Talking points

- After changing the parameter, I rerun the pipeline and check the new metric.
- `dvc metrics diff` is useful because it shows whether the experiment improved or got worse.
- This is the DVC idea of reproducible experimentation in a simple way.


#### Example B: Change the learning rate


In [12]:
epochs: 2
lr: 0.0005
batch_size: 64


#### Talking points

- Here I test a learning rate change while keeping the other settings fixed.
- This helps me see how sensitive the training is to the optimizer step size.
- Again, the important thing is that the change is tracked in a clean and reproducible way.


In [13]:
# Bash or PowerShell

!dvc repro
!dvc metrics show
!dvc metrics diff


Stage 'prepare' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'predict' didn't change, skipping
Data and pipelines are up to date.
Path          accuracy
metrics.json  0.8242


#### Talking points

- I rerun the pipeline after the learning rate update and compare the results.
- If the metric changes, I can connect that change back to the new parameter value.
- This makes the experiment history easier to explain in class or in a submission.


#### Example C: Change the batch size


In [14]:
epochs: 2
lr: 0.001
batch_size: 128


#### Talking points

- This example changes the batch size.
- Batch size affects how many samples are processed at once, so it can change both speed and model behavior.
- Since it is in `params.yaml`, it should be tracked just like the other experiment settings.


In [15]:
# Bash or PowerShell

!dvc repro
!dvc metrics show
!dvc metrics diff


Stage 'prepare' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'predict' didn't change, skipping
Data and pipelines are up to date.
Path          accuracy
metrics.json  0.8242


#### Talking points

- After updating batch size, I rerun the pipeline exactly the same way as before.
- The workflow stays consistent even when the experiment settings change.
- That consistency is one of the main benefits of using DVC in ML projects.


#### Example D: Re-run with no changes

```bash
dvc repro
```

Students should observe that DVC skips unchanged stages. This is a strong demonstration of dependency-aware workflow execution.


### 3.11 Save the project state in Git

In [16]:
# Bash

!git add .gitignore .dvcignore data/.gitignore dvc.yaml dvc.lock params.yaml src/prepare.py src/train.py src/predict.py
!git commit -m "Build DVC pipeline for MNIST workflow"


On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    ML_Workflow_DVC.ipynb
	modified:   README.md

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	ML_Workflow_DVC_updated.ipynb
	data/raw/
	src/__pycache__/

no changes added to commit (use "git add" and/or "git commit -a")


#### Talking points

- These Git commands save the pipeline definition files and source code to the repository.
- I am adding the files that define the workflow, not the large generated artifacts.
- In this workflow, Git stores the project logic and DVC tracks the heavy outputs.


In [17]:
# PowerShell

!git add .gitignore .dvcignore data/.gitignore dvc.yaml dvc.lock params.yaml src/prepare.py src/train.py src/predict.py
!git commit -m "Build DVC pipeline for MNIST workflow"


On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    ML_Workflow_DVC.ipynb
	modified:   README.md

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	ML_Workflow_DVC_updated.ipynb
	data/raw/
	src/__pycache__/

no changes added to commit (use "git add" and/or "git commit -a")


#### Talking points

- This is the same Git step again in a Windows-friendly command block.
- I want the repository to keep the notebook-related source files, YAML files, and DVC metadata.
- That way someone else can clone the project and reproduce the same workflow.


## 4. Talking Points

Use these during the workshop discussion.

### 💡 Talking Point 1
This pipeline is intentionally minimal. The goal is to teach workflow concepts, not advanced model design.

### 💡Talking Point 2
DVC works best when code, parameters, data, and outputs are treated as connected parts of one reproducible system.

### 💡Talking Point 3
Changing a parameter is not just “running training again.” It becomes a tracked, comparable experiment.

### 💡Talking Point 4
When DVC skips a stage, it is showing students that reproducibility is also about **efficiency**.

### 💡Talking Point 5
A weak result in a simple model is still educationally valuable if the workflow is transparent and reproducible.

### 💡Talking Point 6
This small project mirrors a larger professional pattern: data preparation, model training, metrics, and versioned pipeline logic.

### 💡Talking Point 7
Students often think reproducibility starts after model selection. In practice, it starts the moment raw data enters the workflow.

### 💡Talking Point 8
DVC does not replace Git. It extends Git-based collaboration into the data and model parts of machine learning work.


### 🗣️ Additional Talking Points for Section 5

#### 0. Important subtle issue (VERY useful for teaching)
Importing a class from another Python file only works cleanly when that file separates **definitions** from **execution**. If `train.py` runs training code as soon as it is imported, then importing `SimpleCNN` inside prediction code may accidentally trigger training again.

#### 1. Best practice (recommended fix)
A good pattern is to place the training workflow inside a `main()` function and then run it only with:

```python
if __name__ == "__main__":
    main()
```

This allows other files to import `SimpleCNN` safely without causing the training script to execute.

#### 2. Teaching insight (very valuable moment)
This is a useful example of a broader software engineering principle in machine learning: separate **model definitions** from **script execution**. Students often write everything in one file at first, but reusable ML workflows benefit from modular code.

#### 3. Important note about imports in the notebook vs the pipeline
The prediction example in **Section 5.1** can run in the notebook with:

```python
from src.train import SimpleCNN
```

because the notebook is usually executed from the project root, where `src` is visible as a package-like folder.

However, this will not necessarily run as-is when the same code is saved to `src/predict.py` and executed from the pipeline with:

```bash
python src/predict.py
```

For that script-based example, the import may need to be refactored to:

```python
from train import SimpleCNN  # Assuming train.py is in the same directory for this example
```

This distinction is worth highlighting explicitly: **execution context affects how imports work**.


## 5. Expand the Pipeline: Add a Classification Stage

So far, the pipeline prepares data and trains a CNN. A natural next step is to **use the trained model to run classifications** and save prediction outputs.

In this extension, students add a new script called `src/predict.py`. This script loads:

- the trained model from `model.pt`
- the processed test data from `data/processed/test.pt`

It then runs inference and saves a small results file. This demonstrates an important MLOps idea: **training is not the end of the workflow**. In practice, models are trained so they can be used for prediction, evaluation, and downstream decision-making.

### Updated pipeline idea

```text
prepare  →  train  →  predict
```

The new `predict` stage depends on both the trained model and the prediction script. DVC will only rerun this stage when one of its dependencies changes.

This is a good example of how pipelines grow over time: first data preparation, then training, then inference.


### 5.1 Create `src/predict.py`

In [18]:
import os
import json
import torch
import torch.nn as nn

DATA_DIR = "data/processed"
MODEL_PATH = "model.pt"
PREDICTIONS_PATH = "predictions.json"

test_images, test_labels = torch.load(os.path.join(DATA_DIR, "test.pt"))

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(8 * 13 * 13, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = SimpleCNN()
model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device("cpu")))
model.eval()

with torch.no_grad():
    outputs = model(test_images)
    predictions = torch.argmax(outputs, dim=1)

results = []
for i in range(len(predictions)):
    results.append({
        "index": int(i),
        "true_label": int(test_labels[i]),
        "predicted_label": int(predictions[i])
    })

with open(PREDICTIONS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Predictions saved to {PREDICTIONS_PATH}")


Predictions saved to predictions.json


#### Talking points

- This prediction script loads the saved model and the processed test data.
- I kept the model definition inside the script so it does not accidentally rerun training through an import side effect.
- The script writes predictions into a JSON file, which becomes the output of the new DVC stage.


### 5.2 Update `dvc.yaml`

Add a third stage to the pipeline:

```yaml
predict:
  cmd: python src/predict.py
  deps:
    - src/predict.py
    - model.pt
    - data/processed/test.pt
  outs:
    - predictions.json
```

With this addition, DVC understands that prediction depends on the trained model and the processed test data. If the model changes because training is rerun, the `predict` stage will also rerun.


### 5.3 Read a small sample from `predictions.json`

This step is just to quickly verify that the prediction file was created correctly and contains the labels we expect.

In [19]:
import json

with open("predictions.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

predictions[:10]


[{'index': 0, 'true_label': 7, 'predicted_label': 7},
 {'index': 1, 'true_label': 2, 'predicted_label': 2},
 {'index': 2, 'true_label': 1, 'predicted_label': 8},
 {'index': 3, 'true_label': 0, 'predicted_label': 0},
 {'index': 4, 'true_label': 4, 'predicted_label': 4},
 {'index': 5, 'true_label': 1, 'predicted_label': 8},
 {'index': 6, 'true_label': 4, 'predicted_label': 8},
 {'index': 7, 'true_label': 9, 'predicted_label': 9},
 {'index': 8, 'true_label': 5, 'predicted_label': 0},
 {'index': 9, 'true_label': 9, 'predicted_label': 8}]

#### Talking points

- This quick check reads the prediction file and shows only the first few results.
- I do not need to print the entire file just to confirm that the stage worked.
- A small sample is enough to verify the structure and the predicted labels.


### 5.3 Run the expanded pipeline

Once `src/predict.py` and the updated `dvc.yaml` are in place, students can rerun the pipeline from the project root.

They should observe that DVC now executes three stages:

- `prepare`
- `train`
- `predict`

and produces a new output file:

- `predictions.json`

That file contains a small sample of predicted labels from the trained model.


In [20]:
# Bash

!dvc repro
!python -c "import json; print(json.dumps(json.load(open('predictions.json', encoding='utf-8'))[:10], indent=2))"


Stage 'prepare' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'predict' didn't change, skipping
Data and pipelines are up to date.
[
  {
    "index": 0,
    "true_label": 7,
    "predicted_label": 7
  },
  {
    "index": 1,
    "true_label": 2,
    "predicted_label": 2
  },
  {
    "index": 2,
    "true_label": 1,
    "predicted_label": 8
  },
  {
    "index": 3,
    "true_label": 0,
    "predicted_label": 0
  },
  {
    "index": 4,
    "true_label": 4,
    "predicted_label": 4
  },
  {
    "index": 5,
    "true_label": 1,
    "predicted_label": 8
  },
  {
    "index": 6,
    "true_label": 4,
    "predicted_label": 8
  },
  {
    "index": 7,
    "true_label": 9,
    "predicted_label": 9
  },
  {
    "index": 8,
    "true_label": 5,
    "predicted_label": 0
  },
  {
    "index": 9,
    "true_label": 9,
    "predicted_label": 8
  }
]


#### Talking points

- Now the pipeline includes the prediction stage as well.
- When I run `dvc repro`, DVC can decide whether only prediction needs to rerun or whether earlier stages also changed.
- After that, I print a small sample from `predictions.json` to confirm the new stage worked.


In [ ]:
# PowerShell

!dvc repro
!python -c "import json; print(json.dumps(json.load(open('predictions.json', encoding='utf-8'))[:10], indent=2))"


Stage 'prepare' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'predict' didn't change, skipping
Data and pipelines are up to date.
[
  {
    "index": 0,
    "true_label": 7,
    "predicted_label": 7
  },
  {
    "index": 1,
    "true_label": 2,
    "predicted_label": 2
  },
  {
    "index": 2,
    "true_label": 1,
    "predicted_label": 8
  },
  {
    "index": 3,
    "true_label": 0,
    "predicted_label": 0
  },
  {
    "index": 4,
    "true_label": 4,
    "predicted_label": 4
  },
  {
    "index": 5,
    "true_label": 1,
    "predicted_label": 8
  },
  {
    "index": 6,
    "true_label": 4,
    "predicted_label": 8
  },
  {
    "index": 7,
    "true_label": 9,
    "predicted_label": 9
  },
  {
    "index": 8,
    "true_label": 5,
    "predicted_label": 0
  },
  {
    "index": 9,
    "true_label": 9,
    "predicted_label": 8
  }
]


#### Talking points

- This version uses Python to print the prediction sample, so it is more reliable inside a notebook.
- It avoids command differences that sometimes happen between Bash, PowerShell, and Jupyter shells.
- The important thing is still the same: rerun the pipeline and verify the prediction output.


### 5.4 Try a follow-up experiment

Change a value in `params.yaml`, rerun the pipeline, and inspect whether the predictions change.

For example:

- increase `epochs`
- change `lr`
- change `batch_size`

Then run:

```bash
dvc repro
dvc metrics show
```

Students should notice that once training changes, the prediction stage also reruns because it depends on `model.pt`.

This reinforces a key workflow lesson: **pipeline stages are connected by dependencies, not by manual memory**.


## 6. ⚔️ Challenges

These challenges extend the workshop by connecting the DVC-based CNN pipeline to core neural network concepts covered elsewhere in the course. Each challenge includes a coding task and a reflection task.

For each challenge:

1. complete the coding task
2. run the relevant part of the workflow
3. write **at least three talking points** that explain what you observed and why it matters

Your talking points should focus on reasoning, not just reporting a number.


## What you should write for each challenge

For **each** challenge, try to include these four parts in your final notebook submission:

### 1. Code changes
Explain what file you changed and what you added or updated.

### 2. Experiment steps
State what values or options you tested.

### 3. Results
Write the result in simple language. Mention whether performance improved, dropped, or stayed similar.

### 4. Talking points
Write at least **3 short talking points** that sound natural and easy to explain in class.

### 6.1 ⛰️ Challenge: Hyperparameters, Activation Functions, and Initialization

This challenge connects the CNN workflow to activation-function choice, vanishing gradients, and initialization strategy. The slide deck emphasizes comparing gradients for sigmoid and ReLU-family functions, and notes that ReLU is often a strong default for simple tasks while GELU can be stronger on more complex tasks. It also highlights the role of fan-in/fan-out and Glorot/Xavier initialization in stable training.

#### Coding task

Modify `src/train.py` so that the model can be trained with **different activation functions**. For example:

- `ReLU`
- `LeakyReLU`
- `GELU`

Then add an activation setting to `params.yaml`, such as:

```yaml
activation: relu
```

Update the model code so the activation is chosen from the parameter file. After that, run at least **three experiments**, one per activation function.

#### Suggested extension

Add a second parameter for initialization strategy and test one of these:

- default PyTorch initialization
- Xavier/Glorot initialization
- He/Kaiming initialization

#### Commands to run

```bash
dvc repro
dvc metrics show
dvc metrics diff
```

#### Deliverable

Submit:
- the updated code
- the parameter settings you tried
- the resulting metrics

#### Talking points

Write **at least three talking points** that address questions such as:

- Why might sigmoid-like behavior contribute to vanishing gradients in deeper networks?
- Why is ReLU often a practical default?
- In what situations might GELU or LeakyReLU be worth the extra complexity?
- How can initialization affect optimization stability and training speed?


### 6.2 ⛰️ Challenge: Optimizers

This challenge focuses on how optimization changes learning dynamics. The slide deck covers gradients, momentum, Nesterov acceleration, AdaGrad, RMSProp, and Adam, and emphasizes that Adam combines momentum-like behavior with adaptive scaling.

#### Coding task

Modify `src/train.py` so that the optimizer is selected from `params.yaml`. Start by supporting at least:

- `SGD`
- `SGD with momentum`
- `Adam`

Example parameter additions:

```yaml
optimizer: adam
momentum: 0.9
```

Then run at least **three experiments** with the same model and dataset, changing only the optimizer settings. Keep the other hyperparameters fixed so your comparison is meaningful.

#### Suggested extension

If time permits, compare two learning rates for one optimizer and observe whether the optimizer is robust or sensitive to that choice.

#### Commands to run

```bash
dvc repro
dvc metrics show
dvc metrics diff
```

#### Deliverable

Submit:
- the updated optimizer-selection code
- the parameter settings you used
- a short comparison of performance and training behavior

#### Talking points

Write **at least three talking points** that address questions such as:

- How does momentum change the behavior of plain gradient descent?
- Why might Adam converge faster or more smoothly in this project?
- Why is it important to compare optimizers while holding the rest of the setup constant?
- When would a simpler optimizer still be preferable?


### 6.3 ⛰️ Challenge: Forward and Backward Passes

This challenge connects your CNN workflow to the mechanics of neural network learning. The slide deck walks through a small network, its forward pass, the resulting loss, and how changes in weights affect the loss during backpropagation.

#### Coding task

Add instrumentation to `src/train.py` so that, for one mini-batch, you inspect both the **forward pass** and the **backward pass**.

Your code should do all of the following:

1. print or log the model outputs for one batch before the optimizer step
2. compute the loss for that batch
3. call `loss.backward()`
4. inspect at least one gradient tensor, such as:
   - convolution weights
   - fully connected weights
5. perform the optimizer step
6. optionally print the norm of one parameter before and after the update

A simple example of the kind of inspection you might add is:

```python
print(outputs[:5])
print(loss.item())
print(model.conv.weight.grad.norm().item())
```

#### Suggested extension

Save selected gradient statistics to a JSON file so that they become part of the reproducible workflow.

#### Commands to run

```bash
dvc repro
dvc metrics show
```

#### Deliverable

Submit:
- the instrumented training code
- a short explanation of what happened during one forward and backward pass
- any gradient values, norms, or observations you recorded

#### Talking points

Write **at least three talking points** that address questions such as:

- What information is produced in the forward pass?
- What role does the loss play between the forward and backward passes?
- What does a gradient tell you about a parameter update?
- How do forward and backward passes connect the mathematics of learning to the code you executed?


### 6.4 Optional submission format

For each challenge, you may organize your response as:

- **Code changes**
- **Results**
- **At least three talking points**

This format keeps the work aligned with the goals of the workshop: build, run, observe, and explain.


## 7. The Workshop

## What I changed in this notebook version

- added a review summary near the top so the notebook clearly explains what was inconsistent in the project files
- kept the training example aligned with `params.yaml`
- kept `batch_size` inside the DVC parameter example
- cleaned the prediction section so it no longer depends on importing `src.train`
- replaced the weak PowerShell prediction display example with a Python-based one
- cleaned the Git commands so they match a DVC workflow better
- added talking points after every code section so the explanations sound simple, readable, and presentation-friendly
- cleared old notebook outputs so the file is cleaner and easier to rerun from top to bottom


## Final submission checklist

Before submitting, make sure you have:

- completed the notebook from top to bottom
- answered all challenge sections
- added simple talking points
- updated the README with team member names
- pushed the final notebook and related files to GitHub
- copied the repo link for submission


One team member must push the final notebook to GitHub and send the `.git` URL to the instructor before the end of class.

## 🧠 Learning Objectives
- Explain the role of DVC in an MLOps workflow
- Describe the limitations of Git for machine learning projects
- Build and run a multi-stage DVC pipeline
- Track datasets, models, and metrics using DVC
- Modify experiment parameters and compare results
- Understand dependency-aware pipeline execution
- Extend a pipeline to include inference (prediction)
- Apply best practices for modular ML code design
- Differentiate between `git push` and `dvc push`
- Reproduce a workflow on another machine using DVC

## 🧩 Workshop Structure (90 Minutes)
1. **Instructor-led demo of DVC, Hyperparameters, Optimizers,and Forward/Backward Passes** *(20 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(65 min)* – DVC Pipeline and three challenges (work as teams)
3. **Push to GitHub** *(5 min)* – Teams commit and push the one notebook. **Make sure to include your names so it is easy to identify the team that developed the code**.
4. **Instructor Review** - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**
5. **Email Delivery** *(1 min)* – Each team send the instructor an email **with the *.git link** to the GitHub repo **(one email/team)**. Subject on the email is: CSCN8010 - ML Workflow DVC Workshop, Team #_____.


## 💻 Submission Checklist
- ✅ `MK_Workflow_DVC.ipynb` with:
  - Workshop code: Implementing Hyperparameters, Optimizers,and Forward/Backward Passes and DVC orchaestration.
  - **Talking points**: Answers to all talking point questions **in your own words**.
  - Markdown explanations for each major step
the use case that makes use of the knowledge corpus.
- ✅ `README.md` with:
  - ML Pipelines in the context of Machine Learning and their role in the CI/CD world.
  - Team member names
- ✅ GitHub Repo:
  - Public repo named `ML_Workflow_DVC`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**